# Build Stock-Level Panel: Congressional Trading Features

Este notebook construye un panel a nivel (ACCIÓN, MES) con todas las variables de trading congressional.

**Mejoras respecto a versión anterior:**
1. Variables de contexto de mercado (contrarian, high_vol, illiquid, small_cap)
2. Coordinación por comité (committee_coordinated)
3. Señales compuestas adicionales
4. Validación de NaN

---

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Paths
INPUT_TRADES = 'data/outputs/congress_trades_with_committees.parquet'
OUTPUT_DIR = 'data/prediction_bases'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Período de análisis
START_DATE = '2012-01-01'
END_DATE = '2024-12-31'

# Comités con información privilegiada potencial
INFO_COMMITTEES = [
    'Armed Services', 'Financial Services', 'Energy and Commerce',
    'Intelligence', 'Select Committee on Intelligence',
    'Ways and Means', 'Appropriations', 'Health Education Labor and Pensions',
    'Banking, Housing and Urban Affairs', 'Finance',
    'Judiciary', 'Commerce, Science and Transportation'
]

print("="*70)
print("BUILD STOCK-LEVEL PANEL - ENHANCED VERSION")
print("="*70)

BUILD STOCK-LEVEL PANEL - ENHANCED VERSION


## 1. Load Data

In [2]:
df = pd.read_parquet(INPUT_TRADES)
print(f"Trades cargados: {len(df):,}")
print(f"Columnas: {len(df.columns)}")

# Mostrar columnas disponibles
print("\nColumnas disponibles:")
for i, col in enumerate(df.columns):
    print(f"  {i+1:2}. {col}")

Trades cargados: 99,609
Columnas: 97

Columnas disponibles:
   1. Ticker
   2. TickerType
   3. Company
   4. Traded
   5. Transaction
   6. Trade_Size_USD
   7. Status
   8. Subholding
   9. Description
  10. Name
  11. BioGuideID
  12. Filed
  13. Party
  14. District
  15. Chamber
  16. Comments
  17. Quiver_Upload_Time
  18. excess_return
  19. State
  20. last_modified
  21. Ticker_Clean
  22. is_equity
  23. trade_id
  24. return_t
  25. abs_return_t
  26. return_overnight
  27. return_intraday
  28. momentum_5d
  29. momentum_20d
  30. momentum_60d
  31. momentum_252d
  32. realized_vol_30d
  33. parkinson_vol_30d
  34. realized_vol_60d
  35. vol_of_vol_60d
  36. realized_vol_252d
  37. volume_t
  38. dollar_volume_t
  39. volume_ratio_30d
  40. abnormal_volume_30d
  41. amihud_illiq_20d
  42. roll_spread_30d
  43. hl_spread_20d
  44. zero_volume_days_30d
  45. beta_252d
  46. r2_market_252d
  47. alpha_ff3_252d
  48. beta_mkt_ff3_252d
  49. beta_smb_ff3_252d
  50. beta_hml_ff3_

## 2. Basic Cleaning

In [3]:
# Fechas
if 'Traded' in df.columns:
    df['trade_date'] = pd.to_datetime(df['Traded'])
elif 'trade_date' in df.columns:
    df['trade_date'] = pd.to_datetime(df['trade_date'])

if 'Filed' in df.columns:
    df['filed_date'] = pd.to_datetime(df['Filed'])
else:
    df['filed_date'] = df['trade_date']

df['trade_month'] = df['trade_date'].dt.to_period('M')
df['trade_year'] = df['trade_date'].dt.year

# Ticker
if 'Ticker_Clean' in df.columns:
    ticker_col = 'Ticker_Clean'
elif 'Ticker' in df.columns:
    df['Ticker_Clean'] = df['Ticker'].str.upper().str.strip()
    ticker_col = 'Ticker_Clean'
else:
    ticker_col = [c for c in df.columns if 'ticker' in c.lower()][0]

# Nombre del político
name_cols = [c for c in df.columns if c.lower() in ['name', 'full_name', 'politician']]
if not name_cols:
    name_cols = [c for c in df.columns if 'name' in c.lower()]
name_col = name_cols[0] if name_cols else 'Name'

# Filtrar período
df = df[(df['trade_date'] >= START_DATE) & (df['trade_date'] <= END_DATE)]

print(f"Período: {df['trade_date'].min().date()} a {df['trade_date'].max().date()}")
print(f"Trades en período: {len(df):,}")
print(f"Acciones únicas: {df[ticker_col].nunique():,}")
print(f"Políticos únicos: {df[name_col].nunique():,}")

Período: 2012-06-06 a 2024-12-31
Trades en período: 86,624
Acciones únicas: 4,440
Políticos únicos: 296


## 3. Trade-Level Features

In [4]:
# === 3.1 DIRECCIÓN ===
trans_col = 'Transaction' if 'Transaction' in df.columns else None
if trans_col:
    df['is_buy'] = df[trans_col].str.lower().str.contains('purchase|buy', na=False).astype(int)
    df['is_sell'] = df[trans_col].str.lower().str.contains('sale|sell', na=False).astype(int)
else:
    df['is_buy'] = 0
    df['is_sell'] = 0

df['trade_direction'] = df['is_buy'] - df['is_sell']
print(f"Compras: {df['is_buy'].sum():,}, Ventas: {df['is_sell'].sum():,}")

Compras: 43,826, Ventas: 42,388


In [5]:
# === 3.2 MONTO ===
if 'Trade_Size_USD' in df.columns:
    size_map = {
        '$1,001 - $15,000': 8000,
        '$15,001 - $50,000': 32500,
        '$50,001 - $100,000': 75000,
        '$100,001 - $250,000': 175000,
        '$250,001 - $500,000': 375000,
        '$500,001 - $1,000,000': 750000,
        '$1,000,001 - $5,000,000': 3000000,
        'Over $5,000,000': 7500000,
    }
    df['amount_proxy'] = df['Trade_Size_USD'].map(size_map).fillna(8000)
else:
    df['amount_proxy'] = 8000

df['is_large_trade'] = (df['amount_proxy'] >= 100000).astype(int)
print(f"Trades grandes (>=100K): {df['is_large_trade'].mean()*100:.1f}%")

Trades grandes (>=100K): 5.1%


In [6]:
# === 3.3 TIMING ===
df['disclosure_delay'] = (df['filed_date'] - df['trade_date']).dt.days
df['disclosure_delay'] = df['disclosure_delay'].clip(lower=0, upper=365)
df['long_delay'] = (df['disclosure_delay'] > 30).astype(int)

df['day_of_month'] = df['trade_date'].dt.day
df['days_in_month'] = df['trade_date'].dt.daysinmonth
df['end_of_month'] = (df['days_in_month'] - df['day_of_month'] <= 5).astype(int)

df['day_of_week'] = df['trade_date'].dt.dayofweek
df['is_monday'] = (df['day_of_week'] == 0).astype(int)
df['is_friday'] = (df['day_of_week'] == 4).astype(int)

print(f"Disclosure delay promedio: {df['disclosure_delay'].mean():.1f} días")
print(f"Long delay (>30d): {df['long_delay'].mean()*100:.1f}%")

Disclosure delay promedio: 43.7 días
Long delay (>30d): 44.9%


In [7]:
# === 3.4 COMITÉS ===
def is_info_committee(committee_name):
    if pd.isna(committee_name):
        return 0
    for ic in INFO_COMMITTEES:
        if ic.lower() in str(committee_name).lower():
            return 1
    return 0

# Buscar columna de comité
comm_col = None
for col in ['committee_name', 'Committee', 'committee']:
    if col in df.columns:
        comm_col = col
        break

if comm_col:
    df['is_info_committee'] = df[comm_col].apply(is_info_committee)
else:
    df['is_info_committee'] = 0

# Chair/Ranking member
if 'committee_role' in df.columns:
    df['is_chair'] = df['committee_role'].fillna('').str.lower().str.contains('chair|ranking', na=False).astype(int)
else:
    df['is_chair'] = 0

print(f"Trades de comités informativos: {df['is_info_committee'].mean()*100:.1f}%")
print(f"Trades de chairs/ranking: {df['is_chair'].mean()*100:.1f}%")

Trades de comités informativos: 42.0%
Trades de chairs/ranking: 35.9%


In [8]:
# === 3.5 PODER DEL POLÍTICO ===
# Antigüedad
if 'Years in position' in df.columns:
    df['years_in_position'] = pd.to_numeric(df['Years in position'], errors='coerce').fillna(0)
else:
    df['years_in_position'] = 0

df['is_senior'] = (df['years_in_position'] >= 10).astype(int)

# Net worth
if 'Net worth' in df.columns:
    df['net_worth'] = pd.to_numeric(df['Net worth'], errors='coerce').fillna(0)
    median_nw = df.loc[df['net_worth'] > 0, 'net_worth'].median() if (df['net_worth'] > 0).any() else 0
    df['is_wealthy'] = (df['net_worth'] > median_nw).astype(int)
else:
    df['net_worth'] = 0
    df['is_wealthy'] = 0

# Senador
if 'chamber' in df.columns:
    df['is_senator'] = df['chamber'].astype(str).str.lower().str.contains('senate').astype(int)
elif 'Chamber' in df.columns:
    df['is_senator'] = df['Chamber'].astype(str).str.lower().str.contains('senate').astype(int)
else:
    df['is_senator'] = 0

# Power index
df['power_index'] = df['is_chair'] + df['is_senator'] + df['is_senior'] + df['is_info_committee']

print(f"Senadores: {df['is_senator'].mean()*100:.1f}%")
print(f"Senior (>=10 años): {df['is_senior'].mean()*100:.1f}%")
print(f"Power index promedio: {df['power_index'].mean():.2f}")

Senadores: 0.0%
Senior (>=10 años): 5.4%
Power index promedio: 0.83


In [9]:
# === 3.6 PARTIDO ===
party_col = None
for col in ['party', 'Party', 'party_code']:
    if col in df.columns:
        party_col = col
        break

if party_col:
    df['is_democrat'] = df[party_col].astype(str).str.upper().str.contains('D|DEM').astype(int)
    df['is_republican'] = df[party_col].astype(str).str.upper().str.contains('R|REP').astype(int)
else:
    df['is_democrat'] = 0
    df['is_republican'] = 0

print(f"Demócratas: {df['is_democrat'].mean()*100:.1f}%")
print(f"Republicanos: {df['is_republican'].mean()*100:.1f}%")

Demócratas: 49.0%
Republicanos: 99.9%


In [10]:
# === 3.7 COMPORTAMIENTO ===
# Frequent trader
trader_counts = df.groupby(name_col)['trade_date'].count()
frequent_threshold = trader_counts.quantile(0.75)
frequent_traders = trader_counts[trader_counts >= frequent_threshold].index
df['frequent_trader'] = df[name_col].isin(frequent_traders).astype(int)

# First time trading this stock
df = df.sort_values([name_col, ticker_col, 'trade_date'])
df['first_time'] = (~df.duplicated(subset=[name_col, ticker_col], keep='first')).astype(int)

# Direction change
df['prev_is_buy'] = df.groupby([name_col, ticker_col])['is_buy'].shift(1)
df['direction_change'] = ((df['is_buy'] != df['prev_is_buy']) & df['prev_is_buy'].notna()).astype(int)

print(f"Frequent traders: {df['frequent_trader'].mean()*100:.1f}%")
print(f"First time trades: {df['first_time'].mean()*100:.1f}%")
print(f"Direction changes: {df['direction_change'].mean()*100:.1f}%")

Frequent traders: 94.1%
First time trades: 18.5%
Direction changes: 27.3%


In [11]:
# === 3.8 COORDINACIÓN ===
print("Calculando coordinación...")

# Múltiples políticos misma acción mismo día
daily_traders = df.groupby(['trade_date', ticker_col])[name_col].nunique().reset_index()
daily_traders.columns = ['trade_date', ticker_col, 'n_traders_same_day']
df = df.merge(daily_traders, on=['trade_date', ticker_col], how='left')
df['coordinated'] = (df['n_traders_same_day'] >= 2).astype(int)

# Mismo partido
if party_col:
    party_traders = df.groupby(['trade_date', ticker_col, party_col])[name_col].nunique().reset_index()
    party_traders.columns = ['trade_date', ticker_col, party_col, 'n_party_traders']
    df = df.merge(party_traders, on=['trade_date', ticker_col, party_col], how='left')
    df['party_coordinated'] = (df['n_party_traders'] >= 2).astype(int)
else:
    df['party_coordinated'] = 0

# *** NUEVO: Mismo comité mismo día ***
if comm_col:
    comm_traders = df.groupby(['trade_date', ticker_col, comm_col])[name_col].nunique().reset_index()
    comm_traders.columns = ['trade_date', ticker_col, comm_col, 'n_committee_traders']
    df = df.merge(comm_traders, on=['trade_date', ticker_col, comm_col], how='left')
    df['n_committee_traders'] = df['n_committee_traders'].fillna(1)
    df['committee_coordinated'] = (df['n_committee_traders'] >= 2).astype(int)
else:
    df['committee_coordinated'] = 0

print(f"Trades coordinados (general): {df['coordinated'].mean()*100:.1f}%")
print(f"Trades coordinados (partido): {df['party_coordinated'].mean()*100:.1f}%")
print(f"Trades coordinados (comité): {df['committee_coordinated'].mean()*100:.1f}%")

Calculando coordinación...
Trades coordinados (general): 5.9%
Trades coordinados (partido): 1.9%
Trades coordinados (comité): 0.3%


In [12]:
# === 3.9 CONTEXTO DE MERCADO (NUEVAS) ===
print("Calculando variables de contexto de mercado...")

# Verificar qué columnas de mercado existen
mkt_cols = [c for c in df.columns if c.startswith('momentum') or c.startswith('realized_vol') 
            or 'amihud' in c.lower() or c == 'market_cap']
print(f"Columnas de mercado disponibles: {mkt_cols}")

# Contrarian: compra cuando cayó, vende cuando subió
if 'momentum_20d' in df.columns:
    df['contrarian'] = (
        ((df['is_buy'] == 1) & (df['momentum_20d'] < 0)) |
        ((df['is_sell'] == 1) & (df['momentum_20d'] > 0))
    ).astype(int)
    print(f"Trades contrarian: {df['contrarian'].mean()*100:.1f}%")
else:
    df['contrarian'] = 0
    print("⚠️ momentum_20d no disponible, contrarian = 0")

# High volatility: opera cuando vol está alta
if 'realized_vol_30d' in df.columns:
    vol_median = df['realized_vol_30d'].median()
    df['high_vol'] = (df['realized_vol_30d'] > vol_median).astype(int)
    print(f"Trades en alta volatilidad: {df['high_vol'].mean()*100:.1f}%")
else:
    df['high_vol'] = 0
    print("⚠️ realized_vol_30d no disponible, high_vol = 0")

# Illiquid: opera acciones ilíquidas
if 'amihud_illiq_20d' in df.columns:
    illiq_median = df['amihud_illiq_20d'].median()
    df['illiquid'] = (df['amihud_illiq_20d'] > illiq_median).astype(int)
    print(f"Trades en acciones ilíquidas: {df['illiquid'].mean()*100:.1f}%")
else:
    df['illiquid'] = 0
    print("⚠️ amihud_illiq_20d no disponible, illiquid = 0")

# Small cap: opera small caps
if 'market_cap' in df.columns:
    cap_median = df['market_cap'].median()
    df['small_cap'] = (df['market_cap'] < cap_median).astype(int)
    print(f"Trades en small caps: {df['small_cap'].mean()*100:.1f}%")
else:
    df['small_cap'] = 0
    print("⚠️ market_cap no disponible, small_cap = 0")

Calculando variables de contexto de mercado...
Columnas de mercado disponibles: ['momentum_5d', 'momentum_20d', 'momentum_60d', 'momentum_252d', 'realized_vol_30d', 'realized_vol_60d', 'realized_vol_252d', 'amihud_illiq_20d', 'market_cap']
Trades contrarian: 38.6%
Trades en alta volatilidad: 39.6%
Trades en acciones ilíquidas: 39.4%
Trades en small caps: 38.6%


In [13]:
# === 3.10 SEÑALES COMPUESTAS ===
print("Creando señales compuestas...")

# Smart money: compra + info committee + chair
df['smart_money_buy'] = (
    (df['is_buy'] == 1) & 
    (df['is_info_committee'] == 1) & 
    (df['is_chair'] == 1)
).astype(int)

# Smart money sell
df['smart_money_sell'] = (
    (df['is_sell'] == 1) & 
    (df['is_info_committee'] == 1) & 
    (df['is_chair'] == 1)
).astype(int)

# Insider ring: coordinación dentro del mismo comité
df['insider_ring'] = (
    (df['committee_coordinated'] == 1) & 
    (df['is_info_committee'] == 1)
).astype(int)

# Hidden trade: ilíquido/small cap + info committee (más difícil de detectar)
df['hidden_trade'] = (
    ((df['illiquid'] == 1) | (df['small_cap'] == 1)) & 
    (df['is_info_committee'] == 1)
).astype(int)

# Opportunistic: contrarian + large + info committee
df['opportunistic'] = (
    (df['contrarian'] == 1) & 
    (df['is_large_trade'] == 1) & 
    (df['is_info_committee'] == 1)
).astype(int)

print(f"Smart money buys: {df['smart_money_buy'].mean()*100:.2f}%")
print(f"Smart money sells: {df['smart_money_sell'].mean()*100:.2f}%")
print(f"Insider ring: {df['insider_ring'].mean()*100:.2f}%")
print(f"Hidden trades: {df['hidden_trade'].mean()*100:.2f}%")
print(f"Opportunistic: {df['opportunistic'].mean()*100:.2f}%")

Creando señales compuestas...
Smart money buys: 1.25%
Smart money sells: 1.54%
Insider ring: 0.12%
Hidden trades: 19.20%
Opportunistic: 0.40%


## 4. Collapse to Stock-Month Level

In [14]:
print("\nColapsando a nivel (ACCIÓN, MES)...")

# Políticos únicos
unique_politicians = df.groupby([ticker_col, 'trade_month'])[name_col].nunique().reset_index()
unique_politicians.columns = ['ticker', 'month', 'cong_unique_politicians']

# Agregación principal
agg = df.groupby([ticker_col, 'trade_month']).agg(
    # === CONTEOS BÁSICOS ===
    cong_total_trades=('is_buy', 'count'),
    cong_buy_count=('is_buy', 'sum'),
    cong_sell_count=('is_sell', 'sum'),
    
    # === MONTOS ===
    cong_total_amount=('amount_proxy', 'sum'),
    cong_large_trades=('is_large_trade', 'sum'),
    
    # === TIMING ===
    cong_avg_disclosure_delay=('disclosure_delay', 'mean'),
    cong_long_delay_trades=('long_delay', 'sum'),
    cong_end_of_month_trades=('end_of_month', 'sum'),
    cong_monday_trades=('is_monday', 'sum'),
    cong_friday_trades=('is_friday', 'sum'),
    
    # === COMITÉS ===
    cong_info_committee_trades=('is_info_committee', 'sum'),
    cong_chair_trades=('is_chair', 'sum'),
    
    # === PODER ===
    cong_senior_trades=('is_senior', 'sum'),
    cong_wealthy_trades=('is_wealthy', 'sum'),
    cong_senator_trades=('is_senator', 'sum'),
    cong_avg_power_index=('power_index', 'mean'),
    cong_max_power_index=('power_index', 'max'),
    cong_avg_seniority=('years_in_position', 'mean'),
    cong_avg_net_worth=('net_worth', 'mean'),
    
    # === PARTIDO ===
    cong_dem_trades=('is_democrat', 'sum'),
    cong_rep_trades=('is_republican', 'sum'),
    
    # === COMPORTAMIENTO ===
    cong_frequent_trader_trades=('frequent_trader', 'sum'),
    cong_first_time_trades=('first_time', 'sum'),
    cong_direction_change_trades=('direction_change', 'sum'),
    
    # === COORDINACIÓN ===
    cong_coordinated_trades=('coordinated', 'sum'),
    cong_party_coordinated_trades=('party_coordinated', 'sum'),
    cong_committee_coordinated_trades=('committee_coordinated', 'sum'),  # NUEVO
    cong_max_traders_same_day=('n_traders_same_day', 'max'),
    
    # === CONTEXTO DE MERCADO (NUEVAS) ===
    cong_contrarian_trades=('contrarian', 'sum'),
    cong_high_vol_trades=('high_vol', 'sum'),
    cong_illiquid_trades=('illiquid', 'sum'),
    cong_small_cap_trades=('small_cap', 'sum'),
    
    # === SEÑALES COMPUESTAS ===
    cong_smart_money_buy_trades=('smart_money_buy', 'sum'),
    cong_smart_money_sell_trades=('smart_money_sell', 'sum'),
    cong_insider_ring_trades=('insider_ring', 'sum'),
    cong_hidden_trades=('hidden_trade', 'sum'),
    cong_opportunistic_trades=('opportunistic', 'sum'),
    
).reset_index()

agg.columns = ['ticker', 'month'] + list(agg.columns[2:])

# Merge unique politicians
agg = agg.merge(unique_politicians, on=['ticker', 'month'], how='left')

print(f"Panel base: {len(agg):,} observaciones")


Colapsando a nivel (ACCIÓN, MES)...
Panel base: 43,381 observaciones


In [15]:
# === VARIABLES DERIVADAS ===
print("Creando variables derivadas...")

total = agg['cong_total_trades']

# Señal neta
agg['cong_net'] = agg['cong_buy_count'] - agg['cong_sell_count']
agg['cong_buy_ratio'] = agg['cong_buy_count'] / total
agg['cong_csi'] = agg['cong_net'] / total  # Congressional Sentiment Index

# Ratios sobre total
agg['cong_info_ratio'] = agg['cong_info_committee_trades'] / total
agg['cong_chair_ratio'] = agg['cong_chair_trades'] / total
agg['cong_senior_ratio'] = agg['cong_senior_trades'] / total
agg['cong_senator_ratio'] = agg['cong_senator_trades'] / total
agg['cong_dem_ratio'] = agg['cong_dem_trades'] / total
agg['cong_rep_ratio'] = agg['cong_rep_trades'] / total
agg['cong_coordinated_ratio'] = agg['cong_coordinated_trades'] / total
agg['cong_party_coordinated_ratio'] = agg['cong_party_coordinated_trades'] / total
agg['cong_committee_coordinated_ratio'] = agg['cong_committee_coordinated_trades'] / total  # NUEVO
agg['cong_first_time_ratio'] = agg['cong_first_time_trades'] / total
agg['cong_large_ratio'] = agg['cong_large_trades'] / total
agg['cong_long_delay_ratio'] = agg['cong_long_delay_trades'] / total
agg['cong_frequent_trader_ratio'] = agg['cong_frequent_trader_trades'] / total

# Ratios de contexto (NUEVAS)
agg['cong_contrarian_ratio'] = agg['cong_contrarian_trades'] / total
agg['cong_high_vol_ratio'] = agg['cong_high_vol_trades'] / total
agg['cong_illiquid_ratio'] = agg['cong_illiquid_trades'] / total
agg['cong_small_cap_ratio'] = agg['cong_small_cap_trades'] / total

# Intensidad
agg['cong_intensity'] = total / agg['cong_unique_politicians']

# Señales binarias
agg['cong_consensus_buy'] = (agg['cong_buy_ratio'] > 0.7).astype(int)
agg['cong_consensus_sell'] = (agg['cong_buy_ratio'] < 0.3).astype(int)
agg['cong_multiple_politicians'] = (agg['cong_unique_politicians'] > 1).astype(int)
agg['cong_bipartisan'] = ((agg['cong_dem_trades'] > 0) & (agg['cong_rep_trades'] > 0)).astype(int)

# Señales compuestas agregadas
agg['cong_smart_money'] = ((agg['cong_net'] > 0) & 
                           (agg['cong_info_committee_trades'] > 0) & 
                           (agg['cong_chair_trades'] > 0)).astype(int)

agg['cong_strong_buy'] = ((agg['cong_csi'] > 0.5) & 
                          (agg['cong_unique_politicians'] >= 2)).astype(int)

agg['cong_strong_sell'] = ((agg['cong_csi'] < -0.5) & 
                           (agg['cong_unique_politicians'] >= 2)).astype(int)

# NUEVAS señales compuestas
agg['cong_has_insider_ring'] = (agg['cong_insider_ring_trades'] > 0).astype(int)
agg['cong_has_hidden'] = (agg['cong_hidden_trades'] > 0).astype(int)
agg['cong_has_opportunistic'] = (agg['cong_opportunistic_trades'] > 0).astype(int)

panel_cong = agg.copy()

print(f"Panel generado: {len(panel_cong):,} observaciones")
print(f"Variables: {len(panel_cong.columns)}")

Creando variables derivadas...
Panel generado: 43,381 observaciones
Variables: 71


## 5. Validate NaN

In [16]:
# Verificar NaN
print("\n" + "="*50)
print("VALIDACIÓN DE NaN")
print("="*50)

nan_counts = panel_cong.isnull().sum()
nan_cols = nan_counts[nan_counts > 0]

if len(nan_cols) > 0:
    print(f"\n⚠️ Columnas con NaN:")
    print(nan_cols)
    
    # Rellenar NaN en ratios con 0 (si no hubo trades de ese tipo)
    ratio_cols = [c for c in panel_cong.columns if 'ratio' in c]
    for col in ratio_cols:
        panel_cong[col] = panel_cong[col].fillna(0)
    
    # Verificar de nuevo
    nan_counts_after = panel_cong.isnull().sum()
    nan_cols_after = nan_counts_after[nan_counts_after > 0]
    
    if len(nan_cols_after) > 0:
        print(f"\n⚠️ Columnas con NaN después de limpieza:")
        print(nan_cols_after)
    else:
        print("\n✅ Todos los NaN resueltos")
else:
    print("\n✅ No hay NaN en el panel")

print(f"\nNaN totales: {panel_cong.isnull().sum().sum()}")


VALIDACIÓN DE NaN

✅ No hay NaN en el panel

NaN totales: 0


In [17]:
# Verificar infinitos
inf_counts = np.isinf(panel_cong.select_dtypes(include=[np.number])).sum()
inf_cols = inf_counts[inf_counts > 0]

if len(inf_cols) > 0:
    print(f"⚠️ Columnas con infinitos:")
    print(inf_cols)
    
    # Reemplazar infinitos
    for col in inf_cols.index:
        panel_cong[col] = panel_cong[col].replace([np.inf, -np.inf], np.nan)
        panel_cong[col] = panel_cong[col].fillna(0)
    
    print("✅ Infinitos reemplazados con 0")
else:
    print("✅ No hay infinitos en el panel")

✅ No hay infinitos en el panel


## 6. Save

In [18]:
# Guardar
output_path = os.path.join(OUTPUT_DIR, 'panel_congress_stock_month.parquet')
panel_cong.to_parquet(output_path, index=False)
print(f"Guardado: {output_path}")

csv_path = os.path.join(OUTPUT_DIR, 'panel_congress_stock_month.csv')
panel_cong.to_csv(csv_path, index=False)
print(f"Guardado: {csv_path}")

Guardado: data/prediction_bases/panel_congress_stock_month.parquet
Guardado: data/prediction_bases/panel_congress_stock_month.csv


## 7. Summary

In [19]:
print("\n" + "="*70)
print("RESUMEN - PANEL CONGRESSIONAL TRADING")
print("="*70)

cong_vars = [c for c in panel_cong.columns if c.startswith('cong_')]

print(f"""
DIMENSIONES:
  Observaciones:     {len(panel_cong):,}
  Acciones únicas:   {panel_cong['ticker'].nunique():,}
  Meses:             {panel_cong['month'].nunique()}
  Variables:         {len(panel_cong.columns)}
  Variables cong_:   {len(cong_vars)}

CATEGORÍAS DE VARIABLES:
  Conteos básicos:   cong_total_trades, cong_buy_count, cong_sell_count
  Señales:           cong_net, cong_csi, cong_buy_ratio
  Comités:           cong_info_ratio, cong_chair_ratio
  Poder:             cong_senior_ratio, cong_senator_ratio, cong_avg_power_index
  Coordinación:      cong_coordinated_ratio, cong_committee_coordinated_ratio
  Contexto:          cong_contrarian_ratio, cong_high_vol_ratio, cong_illiquid_ratio
  Compuestas:        cong_smart_money, cong_has_insider_ring, cong_has_hidden

NaN totales:         {panel_cong.isnull().sum().sum()}
""")

print("="*70)
print("✅ PANEL COMPLETADO")
print("="*70)


RESUMEN - PANEL CONGRESSIONAL TRADING

DIMENSIONES:
  Observaciones:     43,381
  Acciones únicas:   4,440
  Meses:             150
  Variables:         71
  Variables cong_:   69

CATEGORÍAS DE VARIABLES:
  Conteos básicos:   cong_total_trades, cong_buy_count, cong_sell_count
  Señales:           cong_net, cong_csi, cong_buy_ratio
  Comités:           cong_info_ratio, cong_chair_ratio
  Poder:             cong_senior_ratio, cong_senator_ratio, cong_avg_power_index
  Coordinación:      cong_coordinated_ratio, cong_committee_coordinated_ratio
  Contexto:          cong_contrarian_ratio, cong_high_vol_ratio, cong_illiquid_ratio
  Compuestas:        cong_smart_money, cong_has_insider_ring, cong_has_hidden

NaN totales:         0

✅ PANEL COMPLETADO


In [20]:
# Vista previa
print("\nVISTA PREVIA:")
print(panel_cong.head(10))

print("\nESTADÍSTICAS DE VARIABLES CLAVE:")
key_vars = ['cong_total_trades', 'cong_net', 'cong_csi', 'cong_unique_politicians', 
            'cong_info_ratio', 'cong_avg_power_index', 'cong_contrarian_ratio',
            'cong_committee_coordinated_ratio']
key_vars = [v for v in key_vars if v in panel_cong.columns]
print(panel_cong[key_vars].describe().round(3))


VISTA PREVIA:
     ticker    month  cong_total_trades  cong_buy_count  cong_sell_count  \
0  07/01/28  2019-05                  1               1                0   
1  07/01/36  2019-05                  1               1                0   
2   0QZI.IL  2019-09                  2               2                0   
3  12/01/37  2019-10                  1               1                0   
4   1AMT.MI  2018-07                  1               1                0   
5   1AMT.MI  2018-10                  1               1                0   
6   1AMT.MI  2019-04                  2               0                2   
7   1AMT.MI  2020-03                  2               2                0   
8   1AMT.MI  2020-05                  1               1                0   
9   1AMT.MI  2020-08                  1               1                0   

   cong_total_amount  cong_large_trades  cong_avg_disclosure_delay  \
0             8000.0                  0                       27.0   
1       

In [21]:
# Listar todas las columnas
print("\nTODAS LAS COLUMNAS:")
for i, col in enumerate(panel_cong.columns):
    print(f"  {i+1:2}. {col}")


TODAS LAS COLUMNAS:
   1. ticker
   2. month
   3. cong_total_trades
   4. cong_buy_count
   5. cong_sell_count
   6. cong_total_amount
   7. cong_large_trades
   8. cong_avg_disclosure_delay
   9. cong_long_delay_trades
  10. cong_end_of_month_trades
  11. cong_monday_trades
  12. cong_friday_trades
  13. cong_info_committee_trades
  14. cong_chair_trades
  15. cong_senior_trades
  16. cong_wealthy_trades
  17. cong_senator_trades
  18. cong_avg_power_index
  19. cong_max_power_index
  20. cong_avg_seniority
  21. cong_avg_net_worth
  22. cong_dem_trades
  23. cong_rep_trades
  24. cong_frequent_trader_trades
  25. cong_first_time_trades
  26. cong_direction_change_trades
  27. cong_coordinated_trades
  28. cong_party_coordinated_trades
  29. cong_committee_coordinated_trades
  30. cong_max_traders_same_day
  31. cong_contrarian_trades
  32. cong_high_vol_trades
  33. cong_illiquid_trades
  34. cong_small_cap_trades
  35. cong_smart_money_buy_trades
  36. cong_smart_money_sell_trades